# Importing the Libraries

In [5]:
!pip install xgboost lightgbm catboost

In [4]:
import numpy as np # linear algebra
import pandas as pd  # data loading, processing & manipulation
import seaborn as sns # statistical data visualization
import xgboost as xgb # XGBoost gradient boosting model
import lightgbm as lgb # LightGBM fast gradient boosting model
from scipy.stats import norm # normal distribution & statistical functions
import matplotlib.pyplot as plt # basic plotting & chart customization
from catboost import CatBoostRegressor # CatBoost model for categorical features
from sklearn.metrics import mean_absolute_error   # evaluating model prediction error
from sklearn.model_selection import GridSearchCV  # hyperparameter tuning
from sklearn.model_selection import train_test_split  # splitting data into train & test sets
from sklearn.preprocessing import LabelEncoder, StandardScaler ## encoding categories & scaling features

# Loading the Dataset

In [15]:
df = pd.read_csv('Food_Delivery_Times.csv')
df

,Order_ID,Distance_km,Weather,Traffic_Level,Time_of_Day,Vehicle_Type,Preparation_Time_min,Courier_Experience_yrs,Delivery_Time_min
0,522,7.93,Windy,Low,Afternoon,Scooter,12,1.0,43
1,738,16.42,Clear,Medium,Evening,Bike,20,2.0,84
2,741,9.52,Foggy,Low,Night,Scooter,28,1.0,59
3,661,7.44,Rainy,Medium,Afternoon,Scooter,5,1.0,37
4,412,19.03,Clear,Low,Morning,Bike,16,5.0,68
...,...,...,...,...,...,...,...,...,...
995,107,8.50,Clear,High,Evening,Car,13,3.0,54
996,271,16.28,Rainy,Low,Morning,Scooter,8,9.0,71
997,861,15.62,Snowy,High,Evening,Scooter,26,2.0,81
998,436,14.17,Clear,Low,Afternoon,Bike,8,0.0,55


In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Order_ID                1000 non-null   int64  
 1   Distance_km             1000 non-null   float64
 2   Weather                 970 non-null    object 
 3   Traffic_Level           970 non-null    object 
 4   Time_of_Day             970 non-null    object 
 5   Vehicle_Type            1000 non-null   object 
 6   Preparation_Time_min    1000 non-null   int64  
 7   Courier_Experience_yrs  970 non-null    float64
 8   Delivery_Time_min       1000 non-null   int64  
dtypes: float64(2), int64(3), object(4)
memory usage: 70.4+ KB


In [17]:
df.describe()

,Order_ID,Distance_km,Preparation_Time_min,Courier_Experience_yrs,Delivery_Time_min
count,1000.000000,1000.000000,1000.000000,970.000000,1000.000000
mean,500.500000,10.059970,16.982000,4.579381,56.732000
std,288.819436,5.696656,7.204553,2.914394,22.070915
min,1.000000,0.590000,5.000000,0.000000,8.000000
25%,250.750000,5.105000,11.000000,2.000000,41.000000
50%,500.500000,10.190000,17.000000,5.000000,55.500000
75%,750.250000,15.017500,23.000000,7.000000,71.000000
max,1000.000000,19.990000,29.000000,9.000000,153.000000


# Data Preprocessing

#### Checking for null/missing values

In [18]:
for i in df:
    if df[i].isnull().sum()>0:
        print(f'{i} has:, {df[i].isnull().sum()} missing values.')

Weather has:, 30 missing values.
Traffic_Level has:, 30 missing values.
Time_of_Day has:, 30 missing values.
Courier_Experience_yrs has:, 30 missing values.


### Removing the order_ID column since It has no relationship with delivery time.

In [19]:
df = df.drop("Order_ID", axis=1)

In [20]:
df

,Distance_km,Weather,Traffic_Level,Time_of_Day,Vehicle_Type,Preparation_Time_min,Courier_Experience_yrs,Delivery_Time_min
0,7.93,Windy,Low,Afternoon,Scooter,12,1.0,43
1,16.42,Clear,Medium,Evening,Bike,20,2.0,84
2,9.52,Foggy,Low,Night,Scooter,28,1.0,59
3,7.44,Rainy,Medium,Afternoon,Scooter,5,1.0,37
4,19.03,Clear,Low,Morning,Bike,16,5.0,68
...,...,...,...,...,...,...,...,...
995,8.50,Clear,High,Evening,Car,13,3.0,54
996,16.28,Rainy,Low,Morning,Scooter,8,9.0,71
997,15.62,Snowy,High,Evening,Scooter,26,2.0,81
998,14.17,Clear,Low,Afternoon,Bike,8,0.0,55


In [21]:
df[df.isnull().any(axis=1)]

,Distance_km,Weather,Traffic_Level,Time_of_Day,Vehicle_Type,Preparation_Time_min,Courier_Experience_yrs,Delivery_Time_min
6,9.52,Clear,Low,NaN,Bike,12,1.0,49
14,2.80,Clear,High,Morning,Scooter,10,NaN,33
24,11.20,Clear,Medium,Morning,Bike,23,NaN,73
42,0.99,NaN,Medium,Evening,Bike,15,NaN,32
71,4.17,NaN,Low,Evening,Scooter,5,1.0,22
...,...,...,...,...,...,...,...,...
974,11.68,Clear,NaN,Afternoon,Scooter,25,7.0,70
976,8.96,Snowy,NaN,Morning,Car,6,5.0,51
987,7.44,Rainy,Low,Evening,Bike,27,NaN,53
988,14.39,Rainy,Medium,Morning,Scooter,6,NaN,50


### Handling missing values

In [23]:
categorical_columns = [ 'Weather', 'Traffic_Level', 'Time_of_Day']

for col in categorical_columns:
    df[col] = df[col].fillna(df[col].mode()[0])

In [24]:
df["Courier_Experience_yrs"] = df["Courier_Experience_yrs"].fillna(df["Courier_Experience_yrs"].median())

In [28]:
df.isnull().sum()

Distance_km               0
Weather                   0
Traffic_Level             0
Time_of_Day               0
Vehicle_Type              0
Preparation_Time_min      0
Courier_Experience_yrs    0
Delivery_Time_min         0
dtype: int64

In [30]:
print(f'The number of duplicates value in the data set is: {df.duplicated().sum()}')

The number of duplicates value in the data set is: 0


# Data Vizualization